In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/e-commerce-fraud-detection-dataset/transactions.csv


In [2]:
pip install pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler, LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier, Pool
from pytorch_tabnet.tab_model import TabNetClassifier
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report, average_precision_score

In [4]:
df = pd.read_csv('/kaggle/input/e-commerce-fraud-detection-dataset/transactions.csv')
df

,transaction_id,user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud
0,1,1,141,47,147.93,84.75,FR,FR,web,travel,0,1,1,1,2024-01-06T04:09:39Z,370.95,0
1,2,1,141,47,147.93,107.90,FR,FR,web,travel,0,0,0,0,2024-01-09T20:13:47Z,149.62,0
2,3,1,141,47,147.93,92.36,FR,FR,app,travel,1,1,1,1,2024-01-12T06:20:11Z,164.08,0
3,4,1,141,47,147.93,112.47,FR,FR,web,fashion,0,1,1,1,2024-01-15T17:00:04Z,397.40,0
4,5,1,141,47,147.93,132.91,FR,US,web,electronics,0,1,1,1,2024-01-17T01:27:31Z,935.28,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299690,299691,6000,996,45,27.93,34.07,ES,ES,web,grocery,0,1,1,0,2024-09-29T04:40:54Z,218.55,0
299691,299692,6000,996,45,27.93,68.56,ES,ES,app,travel,0,1,1,1,2024-10-03T08:49:02Z,185.55,0
299692,299693,6000,996,45,27.93,25.02,ES,ES,app,fashion,0,1,1,1,2024-10-26T07:40:38Z,33.50,0
299693,299694,6000,996,45,27.93,23.92,ES,ES,web,gaming,0,0,0,0,2024-10-27T09:31:56Z,71.75,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299695 entries, 0 to 299694
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   transaction_id           299695 non-null  int64  
 1   user_id                  299695 non-null  int64  
 2   account_age_days         299695 non-null  int64  
 3   total_transactions_user  299695 non-null  int64  
 4   avg_amount_user          299695 non-null  float64
 5   amount                   299695 non-null  float64
 6   country                  299695 non-null  object 
 7   bin_country              299695 non-null  object 
 8   channel                  299695 non-null  object 
 9   merchant_category        299695 non-null  object 
 10  promo_used               299695 non-null  int64  
 11  avs_match                299695 non-null  int64  
 12  cvv_result               299695 non-null  int64  
 13  three_ds_flag            299695 non-null  int64  
 14  tran

In [6]:
df.isnull().sum()

transaction_id             0
user_id                    0
account_age_days           0
total_transactions_user    0
avg_amount_user            0
amount                     0
country                    0
bin_country                0
channel                    0
merchant_category          0
promo_used                 0
avs_match                  0
cvv_result                 0
three_ds_flag              0
transaction_time           0
shipping_distance_km       0
is_fraud                   0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df['is_fraud'].value_counts()

is_fraud
0    293083
1      6612
Name: count, dtype: int64

In [9]:
df.drop(columns=['transaction_id', 'user_id'])

,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud
0,141,47,147.93,84.75,FR,FR,web,travel,0,1,1,1,2024-01-06T04:09:39Z,370.95,0
1,141,47,147.93,107.90,FR,FR,web,travel,0,0,0,0,2024-01-09T20:13:47Z,149.62,0
2,141,47,147.93,92.36,FR,FR,app,travel,1,1,1,1,2024-01-12T06:20:11Z,164.08,0
3,141,47,147.93,112.47,FR,FR,web,fashion,0,1,1,1,2024-01-15T17:00:04Z,397.40,0
4,141,47,147.93,132.91,FR,US,web,electronics,0,1,1,1,2024-01-17T01:27:31Z,935.28,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299690,996,45,27.93,34.07,ES,ES,web,grocery,0,1,1,0,2024-09-29T04:40:54Z,218.55,0
299691,996,45,27.93,68.56,ES,ES,app,travel,0,1,1,1,2024-10-03T08:49:02Z,185.55,0
299692,996,45,27.93,25.02,ES,ES,app,fashion,0,1,1,1,2024-10-26T07:40:38Z,33.50,0
299693,996,45,27.93,23.92,ES,ES,web,gaming,0,0,0,0,2024-10-27T09:31:56Z,71.75,0


In [10]:
# create new features
# compare country consistency
df['country_mismatch'] = (df['country'] != df['bin_country']).astype(int)

# ratio between current transaction amount and user's average amount 
df['amount_ratio'] = df['amount'] / (df['avg_amount_user'] + 1e-5)

# reduce skewness of transaction amount
df['log_amount'] = np.log1p(df['amount'])

# time-related features
df['transaction_time'] = pd.to_datetime(df['transaction_time'])
df['hour'] = df['transaction_time'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)

# sort dataset by time
df = df.sort_values(by='transaction_time').reset_index(drop=True)

# drop cols
drop_cols = ['transaction_id', 'user_id', 'transaction_time', 'amount']
df = df.drop(columns= drop_cols)

num_cols = ['account_age_days', 'total_transactions_user', 'shipping_distance_km', 'amount_ratio', 'log_amount', 'avg_amount_user', 'hour', 'hour_sin', 'hour_cos' ]
cat_cols = ['country', 'bin_country', 'merchant_category', 'channel']


X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

# train - test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# apply Robust Scaler
robust_scaler = RobustScaler()
X_train[num_cols] = robust_scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = robust_scaler.transform(X_test[num_cols])

display(X_train)

,account_age_days,total_transactions_user,avg_amount_user,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,shipping_distance_km,country_mismatch,amount_ratio,log_amount,hour,hour_sin,hour_cos
0,0.834802,-1.1,2.668521,PL,PL,web,electronics,0,1,1,0,-0.878854,0,1.685337,1.561604,-1.000000,-8.659561e-17,0.707107
1,-0.411894,-0.7,0.837395,US,US,app,grocery,0,1,1,1,-0.819195,0,-0.388761,0.312658,-1.000000,-8.659561e-17,0.707107
2,0.925110,-0.3,-0.067158,ES,ES,app,electronics,0,1,1,0,0.055490,0,-0.393931,-0.274516,-1.000000,-8.659561e-17,0.707107
3,0.689427,-1.1,1.738970,GB,GB,web,fashion,1,1,1,1,-0.395408,0,0.459431,1.013378,-1.000000,-8.659561e-17,0.707107
4,-0.373348,-0.1,-0.155474,GB,GB,web,gaming,0,1,1,1,0.283106,0,-0.321507,-0.328351,-1.000000,-8.659561e-17,0.707107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239751,0.495595,0.1,-0.167777,PL,PL,web,grocery,0,1,1,1,0.100381,0,1.265817,0.240574,-0.583333,6.830127e-01,0.183013
239752,0.861233,-0.4,0.019513,IT,IT,app,fashion,0,1,1,1,-0.483501,0,0.060373,0.047472,-0.583333,6.830127e-01,0.183013
239753,-0.595815,0.3,-0.427631,NL,NL,app,fashion,0,1,1,1,0.192552,0,0.236546,-0.509607,-0.500000,7.071068e-01,0.000000
239754,-0.865639,0.2,-0.127341,US,US,app,electronics,0,1,1,1,-0.778087,0,1.468976,0.336797,-0.500000,7.071068e-01,0.000000


# **Model Training**

* *Logistic Regression*

In [11]:
# apply OneHotEncoder
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='passthrough'
)

cat_idx = [X_train.columns.get_loc(c) for c in cat_cols]

lr_pipeline = ImbPipeline(steps=[
    ('smote', SMOTENC(
        categorical_features=cat_idx,
        sampling_strategy=0.1,
        random_state=42
    )),
    ('preprocessor', preprocessor_lr),
    ('classifier', LogisticRegression(max_iter=1000, n_jobs=-1))
])

print('Training Logistic Regression')
lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print('AUC-ROC LR:', roc_auc_score(y_test, y_prob_lr))
print('PR-AUC LR:', average_precision_score(y_test, y_prob_lr))
print(classification_report(y_test, y_pred_lr))

Training Logistic Regression
AUC-ROC LR: 0.9437033945577052
PR-AUC LR: 0.6404473885528166
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     58630
           1       0.55      0.60      0.57      1309

    accuracy                           0.98     59939
   macro avg       0.77      0.80      0.78     59939
weighted avg       0.98      0.98      0.98     59939



In [12]:
lr_pipeline2 = ImbPipeline(steps=[
    ('preprocessor', preprocessor_lr),
    ('classifier', LogisticRegression(max_iter=1000, n_jobs=-1, class_weight='balanced'))
])
print('Training Logistic Regression without SMOTE')
lr_pipeline2.fit(X_train, y_train)

y_pred_lr = lr_pipeline2.predict(X_test)
y_prob_lr = lr_pipeline2.predict_proba(X_test)[:, 1]

print('AUC-ROC LR:', roc_auc_score(y_test, y_prob_lr))
print('PR-AUC LR:', average_precision_score(y_test, y_prob_lr))
print(classification_report(y_test, y_pred_lr))

threshold = 0.75
y_pred_new = (y_prob_lr >= threshold).astype(int) 
print(f"Report with threshold {threshold}:")
print(classification_report(y_test, y_pred_new))

Training Logistic Regression without SMOTE
AUC-ROC LR: 0.9489677402289897
PR-AUC LR: 0.641515146563896
              precision    recall  f1-score   support

           0       1.00      0.90      0.95     58630
           1       0.17      0.86      0.28      1309

    accuracy                           0.90     59939
   macro avg       0.58      0.88      0.61     59939
weighted avg       0.98      0.90      0.93     59939

Report with threshold 0.75:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98     58630
           1       0.30      0.75      0.43      1309

    accuracy                           0.96     59939
   macro avg       0.65      0.85      0.71     59939
weighted avg       0.98      0.96      0.97     59939



* *CatBoost*

In [13]:

X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].fillna("Missing").astype(str)
    X_test_cb[col] = X_test_cb[col].fillna("Missing").astype(str)

cb_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_cols,
    auto_class_weights='Balanced', 
    verbose=100,
    eval_metric='AUC'
)

print('Training CatBoost')
cb_model.fit(X_train_cb, y_train, eval_set=(X_test_cb, y_test))

y_prob_cb = cb_model.predict_proba(X_test_cb)[:, 1]
y_pred_cb = cb_model.predict(X_test_cb)

print('AUC-ROC CatBoost:', roc_auc_score(y_test, y_prob_cb))
print(classification_report(y_test, y_pred_cb))

Training CatBoost
0:	test: 0.9515100	best: 0.9515100 (0)	total: 250ms	remaining: 2m 4s
100:	test: 0.9760715	best: 0.9760715 (100)	total: 14.5s	remaining: 57.5s
200:	test: 0.9759065	best: 0.9760755 (180)	total: 26.3s	remaining: 39.1s
300:	test: 0.9763839	best: 0.9764348 (292)	total: 41.2s	remaining: 27.2s
400:	test: 0.9767270	best: 0.9768320 (374)	total: 58.7s	remaining: 14.5s
499:	test: 0.9768844	best: 0.9769181 (471)	total: 1m 16s	remaining: 0us

bestTest = 0.9769180604
bestIteration = 471

Shrink model to first 472 iterations.
AUC-ROC CatBoost: 0.9769180604187777
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     58630
           1       0.33      0.89      0.48      1309

    accuracy                           0.96     59939
   macro avg       0.67      0.93      0.73     59939
weighted avg       0.98      0.96      0.97     59939



* *TabNet*

In [14]:
X_train_tab = X_train.copy()
X_test_tab = X_test.copy()

ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_tab[cat_cols] = ord_encoder.fit_transform(X_train_tab[cat_cols].astype(str))
X_test_tab[cat_cols] = ord_encoder.transform(X_test_tab[cat_cols].astype(str))

for col in cat_cols:
    X_train_tab[col] = X_train_tab[col].astype(int) + 1
    X_test_tab[col] = X_test_tab[col].astype(int) + 1

cat_idxs = [X_train.columns.get_loc(col) for col in cat_cols]
cat_dims = []
for col in cat_cols:
    vocab_size = int(X_train_tab[col].max()) + 1
    cat_dims.append(vocab_size)


for col in X_train.columns:
    if col not in cat_cols:
        med = X_train_tab[col].median()
        X_train_tab[col] = X_train_tab[col].fillna(med)
        X_test_tab[col] = X_test_tab[col].fillna(med)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()

weight_for_1 = neg_count / pos_count 

class_weights = torch.tensor([1.0, weight_for_1], dtype=torch.float32)

print(f"Weights: {class_weights}")

clf_tabnet = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=5,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    verbose=10
)

print('Training TabNet')

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

clf_tabnet.fit(
    X_train=X_train_tab.values, 
    y_train=y_train.values.astype(int), 
    eval_set=[
        (X_train_tab.values, y_train.values.astype(int)), 
        (X_test_tab.values, y_test.values.astype(int))
    ],
    eval_name=['train', 'valid'],
    eval_metric=['auc'],
    loss_fn=loss_fn,
    max_epochs=100,
    patience=10,
    batch_size=1024, 
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

y_pred_tab = clf_tabnet.predict(X_test_tab.values)
y_prob_tab = clf_tabnet.predict_proba(X_test_tab.values)[:, 1]

print('AUC-ROC TabNet:', roc_auc_score(y_test, y_prob_tab))
print(classification_report(y_test, y_pred_tab))

Weights: tensor([ 1.0000, 44.2114])
Training TabNet


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.44703 | train_auc: 0.90393 | valid_auc: 0.90142 |  0:00:23s
epoch 10 | loss: 0.21202 | train_auc: 0.9662  | valid_auc: 0.96018 |  0:04:06s

Early stopping occurred at epoch 15 with best_epoch = 5 and best_valid_auc = 0.96081


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


AUC-ROC TabNet: 0.9608136548465229
              precision    recall  f1-score   support

           0       1.00      0.91      0.95     58630
           1       0.18      0.90      0.30      1309

    accuracy                           0.91     59939
   macro avg       0.59      0.90      0.63     59939
weighted avg       0.98      0.91      0.94     59939

